In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from PyLTSpice import RawRead  # type: ignore

%config InlineBackend.figure_format = 'svg'

In [ ]:
"""
AFE bandwidth step response analysis.
"""

# ============================================
# Load Configuration
# ============================================
with open("config.yaml", "r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

# ============================================
# Output Directory Setup
# ============================================

REPORT_DIR = Path(config["paths"]["report_directory"])
REPORT_DIR.mkdir(exist_ok=True)

print(f"Reports directory ready at: {os.path.abspath(REPORT_DIR)}")

In [ ]:
# ============================================================
# Load RAW File
# ============================================================
raw_path = config["simulation"]["raw_file"]
raw = RawRead(raw_path)

time = np.real(raw.get_trace(config["simulation"]["trace"]["time"]).get_wave(0))
vout_p = raw.get_trace(config["simulation"]["trace"]["output_positive"]).get_wave(0)
vout_n = raw.get_trace(config["simulation"]["trace"]["output_negative"]).get_wave(0)

vdiff = np.real(vout_p - vout_n)

# ------------------------------------------------------------
# Rise Time Measurement (10% to 90%) - Polarity Independent
# ------------------------------------------------------------
v_start = vdiff[0]
v_end = np.mean(vdiff[-100:])

delta = v_end - v_start

v_10 = v_start + 0.1 * delta
v_90 = v_start + 0.9 * delta

t_10 = None
t_90 = None

for i, value in enumerate(vdiff):
    if delta > 0:
        if t_10 is None and value >= v_10:
            t_10 = time[i]
        if t_90 is None and value >= v_90:
            t_90 = time[i]
            break
    else:
        if t_10 is None and value <= v_10:
            t_10 = time[i]
        if t_90 is None and value <= v_90:
            t_90 = time[i]
            break

if t_10 is not None and t_90 is not None and (t_90 > t_10):
    rise_time_s = t_90 - t_10
    bandwidth_est_hz = 0.35 / rise_time_s
else:
    rise_time_s = None
    bandwidth_est_hz = None

# ============================================================
# Plot
# ============================================================
plt.figure(figsize=(10, 6))
plt.plot(time * 1e9, vdiff)
plt.xlabel("Time [ns]")
plt.ylabel("Differential Output [V]")
plt.title("AFE Bandwidth Step Response")
plt.grid(True)
plt.tight_layout()

plt.savefig(REPORT_DIR / "AFEBandwidthStepResponse.svg", format="svg")
plt.show()

# ============================================================
# Summary
# ============================================================
print("\n========== BANDWIDTH TRANSIENT REPORT ==========")

if rise_time_s is not None:
    print(f"Rise Time (10-90%):        {rise_time_s * 1e9:.2f} ns")
    print(f"Estimated -3 dB BW:        {bandwidth_est_hz / 1e6:.2f} MHz")
else:
    print("Rise Time:                 Could not determine")

print("================================================\n")